# 🎮 Нативный Linux Sirus WoW Launcher в Google Colab с трансляцией на телефон! 📱🖥️

Этот блокнот запускает официальную **нативную Linux-версию Sirus Launcher (World of Warcraft)** напрямую (без Wine), используя оригинальный, проверенный метод трансляции и туннелирования, который работает без сбоев.

### ⚡ Особенности:
- **Нативный Linux Sirus Launcher:** Работает быстро и плавно.
- **Оригинальный стриминг:** Использует оригинальные проверенные команды VNC и Cloudflare Tunnel, которые работали в первой версии.
- **Поддержка Прокси-обхода (Replit/SOCKS5):** Если Google Colab блокирует торрент-запросы лаунчера, вы можете пустить трафик через свой прокси на Replit!

---
## 🛠️ Шаг 1: Установка системных библиотек и скачивание Sirus Launcher (Linux)
Нажмите на кнопку запуска (Play) ниже, чтобы установить все графические библиотеки и скачать нативную Linux-версию Sirus.

In [ ]:
#@title Нажмите Play для автоматической установки { display-mode: "form" }

import os
from IPython.display import clear_output

print("🔄 1/4 Обновление репозиториев Linux...")
os.system("apt-get update -qq")

print("🔄 2/4 Установка графических и системных библиотек рабочего стола...")
os.system("apt-get install -y --no-install-recommends "
          "xvfb x11vnc fluxbox wget curl git python3-pip "
          "libnss3 libnspr4 libatk1.0-0 libatk-bridge2.0-0 "
          "libcups2 libdrm2 libgtk-3-0 libgbm1 libasound2 "
          "libxshmfence1 libxkbcommon-x11-0")

print("🔄 3/4 Скачивание и распаковка нативного Sirus Launcher для Linux...")
if not os.path.exists('sirus_launcher.AppImage'):
    os.system("wget -q -O sirus_launcher.AppImage https://sirus.su/launcher/latest-linux")
    os.system("chmod +x sirus_launcher.AppImage")
    os.system("./sirus_launcher.AppImage --appimage-extract")

print("🔄 4/4 Настройка прокси-транслятора noVNC...")
if not os.path.exists('/opt/noVNC'):
    os.system("git clone --depth 1 https://github.com/novnc/noVNC.git /opt/noVNC")
if not os.path.exists('/opt/noVNC/utils/websockify'):
    os.system("git clone --depth 1 https://github.com/novnc/websockify /opt/noVNC/utils/websockify")

clear_output()
print("✅ Шаг 1 успешно завершен! Все системные файлы и Linux-лаунчер Сируса готовы к работе.")

---
## 🖥️ Шаг 2: Запуск виртуального экрана, VNC и оригинального noVNC
Запуск транслятора noVNC по оригинальной схеме, которая успешно работала в первой версии.

In [ ]:
#@title Нажмите Play для оригинального запуска трансляции { display-mode: "form" }

import os
import time
import subprocess

print("🧹 1. Очистка старых процессов...")
os.system("pkill -9 -f Xvfb")
os.system("pkill -9 -f fluxbox")
os.system("pkill -9 -f x11vnc")
os.system("pkill -9 -f novnc_proxy")
os.system("pkill -9 -f websockify")
os.system("pkill -9 -f siruslauncher")
time.sleep(1)

print("🧹 2. Удаление блокировок экрана...")
os.system("rm -f /tmp/.X99-lock")
os.system("rm -f /tmp/.X11-unix/X99")
time.sleep(0.5)

print("🖥️ 3. Запуск виртуального дисплея (:99)...")
subprocess.Popen("Xvfb :99 -screen 0 1280x720x24", shell=True)
time.sleep(1.5)

print("🎨 4. Запуск Fluxbox и оригинального x11vnc...")
subprocess.Popen("DISPLAY=:99 fluxbox", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.Popen("DISPLAY=:99 x11vnc -forever -nopw -listen localhost -xkb", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1.5)

print("🔄 5. Запуск оригинального noVNC прокси...")
subprocess.Popen("/opt/noVNC/utils/novnc_proxy --vnc localhost:5900 --listen 6080", shell=True)
time.sleep(2.5)

print("✅ Шаг 2 успешно завершен! Стрим-сервер запущен по оригинальной схеме.")

---
## 🌐 Шаг 3: Оригинальный запуск туннеля Cloudflare
Запускает туннель Cloudflare и парсит ссылку оригинальным проверенным скриптом.

In [ ]:
#@title Нажмите Play для создания туннеля { display-mode: "form" }

import os
import subprocess
import re
import time
import urllib.parse
from IPython.display import display, HTML

print("🔄 Запуск Cloudflare Tunnel...")

# Установка cloudflared при необходимости
if not os.path.exists('/usr/local/bin/cloudflared') and not os.path.exists('/usr/bin/cloudflared'):
    print("📥 Скачивание и установка утилиты cloudflared...")
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

# Очистка старых туннелей
os.system("pkill -f cloudflared")
time.sleep(1)

# Запуск нового туннеля в фоне (оригинальный метод)
cmd = "cloudflared tunnel --url http://127.0.0.1:6080"
process = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url = None
print("🔍 Ожидание создания публичной безопасной ссылки...")
for i in range(40):
    line = process.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        # Оптимальные параметры для телефона: автоподключение и автомасштабирование экрана
        vnc_url = f"{url}/vnc.html?autoconnect=true&resize=scale"
        
        from IPython.display import clear_output
        clear_output()
        
        # Создаем QR-код с помощью открытого API
        qr_api_url = f"https://api.qrserver.com/v1/create-qr-code/?size=250x250&data={urllib.parse.quote(vnc_url)}"
        
        html_code = f"""
        <div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; padding: 25px; border: 3px dashed #4CAF50; border-radius: 15px; background-color: #fcfcfc; max-width: 550px; margin: 20px auto; text-align: center; box-shadow: 0 4px 15px rgba(0,0,0,0.1);">
            <h2 style="color: #4CAF50; margin-top: 0; font-size: 26px;">🎉 Стрим Сируса Готов! 🎉</h2>
            <p style="font-size: 16px; margin: 10px 0; color: #333;">Откройте эту ссылку на вашем телефоне или компьютере:</p>
            <p style="font-weight: bold; font-size: 20px; margin: 20px 0;">
                <a href="{vnc_url}" target="_blank" style="color: #ffffff; background-color: #4CAF50; padding: 12px 25px; border-radius: 8px; text-decoration: none; display: inline-block; box-shadow: 0 3px 6px rgba(76,175,80,0.4);">👉 Открыть Трансляцию Экрана 👈</a>
            </p>
            <p style="color: #666; font-size: 15px;">Или просто отсканируйте этот QR-код камерой мобильного:</p>
            <div style="margin: 25px 0;">
                <img src="{qr_api_url}" alt="QR Code" style="border: 3px solid #eee; padding: 8px; background: white; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.05);" />
            </div>
        </div>
        """
        display(HTML(html_code))
        break
    time.sleep(0.1)

if not url:
    print("❌ Ошибка: не удалось получить ссылку туннеля. Пожалуйста, перезапустите эту ячейку.")

---
## 🚀 Шаг 4: Подключение папки скачивания и запуск (С поддержкой прокси Replit)
Эта ячейка автоматически выключает лаунчер, прописывает путь к игре `/content/sirus-wow`, а также позволяет **перенаправить торрент-трафик лаунчера через ваш прокси на Replit**, полностью скрыв его от блокировок Google Colab! 

In [ ]:
replit_socks5_proxy = "" #@param {type:"string"}
#@markdown 🔌 **Пример формата прокси:** `socks5://имя-проекта.имя-пользователя.repl.co:1080` (оставьте пустым, если запускаете без прокси)

import os
import json
import time
import subprocess

print("🧹 1. Принудительно выключаем лаунчер для изменения конфигураций...")
os.system("pkill -9 -f siruslauncher")
os.system("pkill -9 -f AppRun")
time.sleep(1)

# Создаем папку для игры
os.makedirs("/content/sirus-wow", exist_ok=True)

config_dirs = [
    os.path.expanduser("~/.config/Sirus Launcher"),
    os.path.expanduser("~/.config/sirus-open-launcher"),
    os.path.expanduser("~/.config/sirus-launcher"),
    os.path.expanduser("~/.config/SirusLauncher"),
    os.path.expanduser("~/.config/siruslauncher")
]

print("⚙️ 2. Прописываем путь '/content/sirus-wow' в файлы конфигураций...")
found = False
for d in config_dirs:
    os.makedirs(d, exist_ok=True)
    for filename in ["config.json", "settings.json", "sirus.json"]:
        file_path = os.path.join(d, filename)
        try:
            if os.path.exists(file_path):
                with open(file_path, "r", encoding="utf-8") as f:
                    data = json.load(f)
            else:
                data = {}
        except Exception:
            data = {}
        
        data['gamePath'] = "/content/sirus-wow"
        data['path'] = "/content/sirus-wow"
        data['gamedir'] = "/content/sirus-wow"
        data['game-path'] = "/content/sirus-wow"
        data['gameDir'] = "/content/sirus-wow"
        
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        found = True

print("✅ Путь к папке игры успешно зафиксирован!")
time.sleep(0.5)

# Настраиваем переменные окружения прокси для Electron / WebTorrent
if replit_socks5_proxy:
    os.environ['ALL_PROXY'] = replit_socks5_proxy
    os.environ['socks_proxy'] = replit_socks5_proxy
    os.environ['http_proxy'] = replit_socks5_proxy
    os.environ['https_proxy'] = replit_socks5_proxy
    print(f"🔌 Весь трафик лаунчера перенаправлен через прокси Replit: {replit_socks5_proxy}")
else:
    # Сбрасываем прокси при очистке поля
    os.environ.pop('ALL_PROXY', None)
    os.environ.pop('socks_proxy', None)
    os.environ.pop('http_proxy', None)
    os.environ.pop('https_proxy', None)

print("🚀 3. Запускаем нативный Linux Sirus Launcher...")
cmd = "DISPLAY=:99 ./squashfs-root/siruslauncher --no-sandbox --disable-gpu --disable-software-rasterizer"
subprocess.Popen(cmd, shell=True)

print("\n✨ ГОТОВО! Переключитесь на экран телефона. Лаунчер запустится уже с подключенной папкой и прокси!")

---
## 🛡️ Шаг 5: Бесшумная защита от отключения Google Colab (Анти-АФК)
Запустите эту ячейку, чтобы сессия Google Colab не останавливалась сама по себе, когда вы сворачиваете вкладку или выключаете телефон. Она работает абсолютно бесшумно, без вывода лишнего текста на экран.

In [ ]:
#@title Нажмите Play для бесшумного удержания сессии { display-mode: "form" }

import time

print("🛡️ Бесшумное удержание сессии Colab запущено (активно в течение 6 часов).")
print("Экран будет оставаться чистым. Вы можете спокойно играть на телефоне!")

for _ in range(360): # 6 часов
    time.sleep(60)